# ML-07 — Baseline Action Score and Top-20 Review

[w04_baseline_score.ipynb](file:///c:/Users/Rida%20Eman/Downloads/Flyrank%20AI_intenship/work/notebooks/w04_baseline_score.ipynb)

This notebook constructs a transparent baseline scoring rule based on search visibility and page age, evaluates its performance, writes the baseline refresh queue CSV, and inspects the top 20 recommendations.

## 1. My rule and its reason codes

**Baseline Rule**: A page receives a baseline refresh priority score if it ranks on page 1 or 2 (`avg_position <= 20`) and has not been updated in over 180 days (`content_age_days >= 180`).

**Reason Codes**:
- `stale_visible_page`: `content_age_days >= 180` and `impressions_90d >= 500`.
- `low_ctr_visible_page`: `avg_position <= 20` and `ctr < 0.5`.
- `page_one_decay_risk`: `avg_position <= 10` and `content_age_days >= 180`.

In [2]:
import pandas as pd
import numpy as np
import os

# Load starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['avg_position_clean'] = df['avg_position'].fillna(15.0)
df['ctr_clean'] = df['ctr'].fillna(0.0)

# Compute baseline score (0 to 100)
df['baseline_score'] = (
    0.4 * (df['impressions_90d'] / (df['impressions_90d'].max() + 1e-5)) +
    0.3 * (df['content_age_days'] >= 180).astype(int) +
    0.3 * (df['avg_position_clean'] <= 20).astype(int)
) * 100

def assign_reason_code(row):
    if row['content_age_days'] >= 180 and row['impressions_90d'] >= 500:
        return 'stale_visible_page'
    elif row['avg_position_clean'] <= 20 and row['ctr_clean'] < 0.5:
        return 'low_ctr_visible_page'
    elif row['avg_position_clean'] <= 10 and row['content_age_days'] >= 180:
        return 'page_one_decay_risk'
    return 'general_stale'

df['reason_code'] = df.apply(assign_reason_code, axis=1)
print("Baseline score distribution:")
print(df['baseline_score'].describe())

Baseline score distribution:
count    30000.000000
mean        39.848794
std         18.979427
min          0.000077
25%         30.015143
50%         30.393692
75%         60.021247
max        100.000000
Name: baseline_score, dtype: float64


## 2. Build the ranked queue (writes the CSV)

We rank all pages by `baseline_score` descending and export the top queue to `outputs/baseline_action_score.csv`.

In [4]:
os.makedirs("../../outputs", exist_ok=True)
ranked_df = df.sort_values(by='baseline_score', ascending=False)[['content_id', 'client_id', 'baseline_score', 'reason_code', 'trend_direction']]
ranked_df.to_csv("../../outputs/baseline_action_score.csv", index=False)
print(f"Exported {len(ranked_df):,} ranked baseline records to outputs/baseline_action_score.csv.")

Exported 30,000 ranked baseline records to outputs/baseline_action_score.csv.


## 3. Top-20 review

We inspect the top 20 baseline picks to verify that reason codes match the page metrics.

In [6]:
top_20 = ranked_df.head(20)
print("Top 20 Baseline Candidates:")
print(top_20.to_string(index=False))

Top 20 Baseline Candidates:
          content_id         client_id  baseline_score        reason_code trend_direction
content_5fe46e04994d client_4e07408562      100.000000 stale_visible_page            down
content_aaef01a50def client_19581e27de       99.953179 stale_visible_page          stable
content_8c19996aa890 client_4e07408562       99.346127 stale_visible_page            down
content_4c36c775b818 client_4e07408562       95.780536 stale_visible_page            down
content_1a9e894be2e2 client_19581e27de       92.155143 stale_visible_page            down
content_2c2606c5d176 client_19581e27de       86.840945 stale_visible_page            down
content_db5989a78dd3 client_4e07408562       86.664169 stale_visible_page              up
content_9532f197bbc8 client_4e07408562       83.888974 stale_visible_page            down
content_8e7ba84a972b client_7f2253d7e2       82.284539 stale_visible_page          stable
content_8451fc6f034d client_d029fa3a95       81.026549 stale_visible_pag

## 4. Weak picks + leakage check

**Weak Picks**: Fixed age cutoffs flag old pages that are still gaining impressions (e.g. evergreen content), leading to unnecessary refresh recommendations.

**Leakage Audit**: Confirmed that `trend_direction` and `trend_pct` were NOT used in calculating the baseline score.

In [8]:
# Confirm baseline evaluation metrics
is_declining = (df['trend_direction'] == 'down').astype(int)
p50 = (df.sort_values(by='baseline_score', ascending=False).head(50)['trend_direction'] == 'down').mean()
print(f"Baseline Precision@50 on starter dataset: {p50:.4f}")

Baseline Precision@50 on starter dataset: 0.3800


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.